# SafeSound 기준선 학습 (v1)

**셀을 위에서부터 순서대로 실행한다.** 셀 2는 런타임을 재시작시키므로, 재시작 후
셀 3부터 이어서 누르면 된다.

- 데이터: `dataset-v1` 태그 (창 13,007 / 5,207, int8 1.2초 저장)
- 모델: KWS20 v3 파생 1D CNN, 5클래스, QAT 8bit
- 산출물 1순위는 **혼동행렬**이다 — 배경음이 이벤트 합의 2.8배라 전체 정확도는
  전부 배경음으로 찍어도 74%가 나온다

> 런타임 유형을 **GPU (T4)** 로 먼저 바꿀 것. 수정 → 노트 설정 → 하드웨어 가속기.

## 0. Drive 에 올려둬야 하는 파일

| 파일 | 내용 |
|---|---|
| `MyDrive/max78000/safesound-v1.tar.gz` + `.sha256` | 데이터셋 (140MB) |
| `MyDrive/max78000/safesound-code-v1.tar.gz` + `.sha256` | 우리 코드 (수백 KB — 크기는 커밋마다 달라진다) |
| `MyDrive/max78000/ai8x-conda-env.tar.gz` *(선택)* | 환경 백업 3.2GB — 있으면 셀 3이 15분 아낀다 |

없으면 셀 3이 환경을 처음부터 만든다.

In [ ]:
# 1. Drive 마운트 — 데이터·코드·체크포인트가 모두 여기에 있다
import os

from google.colab import drive

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/max78000'
need = ['safesound-v1.tar.gz', 'safesound-v1.tar.gz.sha256',
        'safesound-code-v1.tar.gz', 'safesound-code-v1.tar.gz.sha256']
for f in need:
    print(('있음    ' if os.path.exists(f'{BASE}/{f}') else '없음 ✗  ') + f)
print(('있음    ' if os.path.exists(f'{BASE}/ai8x-conda-env.tar.gz')
       else '없음(새로 빌드)  ') + 'ai8x-conda-env.tar.gz')
missing = [f for f in need if not os.path.exists(f'{BASE}/{f}')]
assert not missing, f'Drive 에 없다: {missing}'

In [ ]:
# 2. condacolab 설치 — ai8x 는 Python 3.11 을 요구하는데 Colab 기본은 3.13 이다 (런타임 재시작됨)
!pip install -q condacolab
import condacolab

condacolab.install()

In [ ]:
# 3. ai8x 환경 준비 — Drive 백업이 있으면 복원, 없으면 CLAUDE.md 12장 해결책을 적용해 새로 만든다
import os
import subprocess

BASE = '/content/drive/MyDrive/max78000'
ENV = '/usr/local/envs/ai8x'


def sh(cmd, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode:
        raise SystemExit(f'실패: {cmd}')
    return r.returncode


# 어느 경로든 필요하다: torchaudio 의 speed_augment 가 시스템 libsox 를 쓴다.
# conda 환경 밖이라 환경 백업에 포함되지 않는다 (CLAUDE.md 12장)
sh('apt-get -qq install -y libsox3 libsox-dev libsox-fmt-all sox')
# Colab 의 matplotlib_inline 백엔드가 conda 환경에 없다
os.environ['MPLBACKEND'] = 'Agg'

if os.path.exists(ENV):
    print('환경이 이미 있다')
elif os.path.exists(f'{BASE}/ai8x-conda-env.tar.gz'):
    print('Drive 백업에서 복원한다 (약 3분)')
    sh(f'mkdir -p {ENV}')
    sh(f'tar xzf {BASE}/ai8x-conda-env.tar.gz -C /usr/local/envs')
else:
    print('환경을 새로 만든다 (약 15~20분)')
    sh('conda create -y -n ai8x python=3.11')
    # torch 2.3.1 은 PyPI 에서 내려갔다 → PyTorch 공식 cu121 인덱스에서 받는다
    sh('conda run -n ai8x --no-capture-output pip install -q '
       'torch==2.3.1 torchvision==0.18.1 '
       '--index-url https://download.pytorch.org/whl/cu121')

print()
sh('conda run -n ai8x --no-capture-output python -c '
   '"import sys, torch; print(sys.version.split()[0], torch.__version__, '
   'torch.cuda.is_available())"')

In [ ]:
# 4. ai8x-training 을 받고 CLAUDE.md 12장의 패치를 적용한다 (distiller 충돌·예외 처리)
import os
import subprocess


def sh(cmd, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode:
        raise SystemExit(f'실패: {cmd}')
    return r.returncode


if not os.path.exists('/content/ai8x-training'):
    sh('git clone -q --recursive '
       'https://github.com/analogdevicesinc/ai8x-training.git /content/ai8x-training')

os.chdir('/content/ai8x-training')

# (1) distiller 디렉터리명이 네임스페이스 패키지 충돌을 만든다 —
#     /content/ai8x-training/distiller 가 설치된 패키지를 가린다
if os.path.isdir('distiller') and not os.path.isdir('distiller_repo'):
    sh('mv distiller distiller_repo')

# (2) distiller/__init__.py 의 except pkg_resources.DistributionNotFound 는
#     ContextualVersionConflict 를 못 잡아 모듈 로딩이 중간에 끊긴다.
#     knowledge_distillation 없음 / __file__ None / __version__ 없음 의 공통 원인
init = 'distiller_repo/distiller/__init__.py'
if os.path.exists(init):
    s = open(init, encoding='utf-8').read()
    if 'except pkg_resources.DistributionNotFound' in s:
        s = s.replace('except pkg_resources.DistributionNotFound', 'except Exception')
        open(init, 'w', encoding='utf-8').write(s)
        print('distiller __init__ 예외 처리 패치 적용')

# 환경 백업을 복원한 경우 이미 설치돼 있다. 실패해도 진행한다 (check=False)
sh('conda run -n ai8x --no-capture-output pip install -q -r requirements.txt', check=False)
sh('conda run -n ai8x --no-capture-output pip install -q -e distiller_repo', check=False)

# 로그·체크포인트는 Drive 에 둔다 — 세션이 끊겨도 남고, 재실행 시 여기서 재개한다
sh('rm -rf logs')
sh('mkdir -p /content/drive/MyDrive/max78000/logs')
sh('ln -s /content/drive/MyDrive/max78000/logs logs')
print()
print('logs ->', os.path.realpath('logs'))

In [ ]:
# 5. 우리 코드 tar 를 해시 확인 후 풀고, ai8x 쪽에 심볼릭 링크를 건다
import hashlib
import json
import os
import tarfile
import time

BASE = '/content/drive/MyDrive/max78000'


def verify(path, manifest=None):
    '''해시를 확인하고, 틀리면 **어느 파일이 옛것인지** 알려준다.

    예전에는 "불일치 ✗" 만 찍고 죽어서, tar 가 옛것인지 .sha256 만 옛것인지
    구별할 수 없었다. Drive 업로드는 둘 중 하나만 갱신되는 일이 흔하다
    (같은 이름을 덮어쓰지 않고 "사본"으로 추가하는 경우도 있다).
    '''
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for c in iter(lambda: f.read(1 << 20), b''):
            h.update(c)
    got = h.hexdigest()
    want = open(path + '.sha256').read().split()[0]
    name = os.path.basename(path)
    if got == want:
        print(f'{name}  OK  {got[:16]}...  {os.path.getsize(path):,} B')
        return
    print(f'{name}  불일치 ✗')
    print(f'  tar 실측   {got}')
    print(f'  사이드카   {want}')
    print(f'  tar 크기   {os.path.getsize(path):,} B')
    print(f'  tar mtime  {time.ctime(os.path.getmtime(path))}')
    print(f'  .sha256 mtime {time.ctime(os.path.getmtime(path + ".sha256"))}')
    if manifest:
        # tar 안의 커밋을 읽으면 어느 버전이 Drive 에 있는지 확정된다
        try:
            with tarfile.open(path) as tf:
                m = json.load(tf.extractfile(manifest))
            print(f'  tar 안의 커밋 {m["commit"][:9]} '
                  f'({len(m["files"])}개 경로, {m["created"]})')
        except Exception as e:
            print(f'  tar 를 열 수 없다 ({e}) → 업로드가 잘린 것이다')
    sib = sorted(f for f in os.listdir(os.path.dirname(path))
                 if f.startswith(os.path.splitext(name)[0][:14]))
    print(f'  같은 폴더의 비슷한 이름: {sib}')
    print('  판독: 실측이 기대값과 같은데 사이드카만 다르면 .sha256 만 다시')
    print('        올리면 된다. 실측·크기가 둘 다 다르면 tar 가 옛것이다.')
    print('        "사본" 파일이 보이면 덮어쓰기가 안 된 것이다.')
    raise AssertionError(f'{path} — tar 와 .sha256 이 서로 다른 버전이다')


verify(f'{BASE}/safesound-code-v1.tar.gz', manifest='CODE_MANIFEST.json')
os.makedirs('/content/max78000-proj', exist_ok=True)
with tarfile.open(f'{BASE}/safesound-code-v1.tar.gz') as tf:
    tf.extractall('/content/max78000-proj')

# 벤더 레포는 고치지 않고 링크만 건다 (CLAUDE.md 11장)
# melfeat.py / safesound_mel.py / ai85net-safesound-mel.py 는 G8 구성 ① 용이다.
for rel in ['datasets/safesound.py', 'models/ai85net-safesound.py',
            'datasets/safesound_wave2d.py',
            'datasets/melfeat.py', 'datasets/safesound_mel.py',
            'models/ai85net-safesound-mel.py']:
    dst = f'/content/ai8x-training/{rel}'
    if os.path.islink(dst) or os.path.exists(dst):
        os.remove(dst)
    os.symlink(f'/content/max78000-proj/{rel}', dst)
    print('link', dst, '->', os.readlink(dst))

print()
print('코드 번들 커밋:',
      json.load(open('/content/max78000-proj/CODE_MANIFEST.json'))['commit'][:9])

In [ ]:
# 6. 데이터셋 해시 확인 후 압축 해제 — 경로는 data/SafeSound/{train,test}/{클래스}/ 가 된다
import hashlib
import json
import os
import tarfile

BASE = '/content/drive/MyDrive/max78000'
DATA = '/content/ai8x-training/data/SafeSound'

want = open(f'{BASE}/safesound-v1.tar.gz.sha256').read().split()[0]
h = hashlib.sha256()
with open(f'{BASE}/safesound-v1.tar.gz', 'rb') as f:
    for c in iter(lambda: f.read(1 << 20), b''):
        h.update(c)
got = h.hexdigest()
if got != want:
    # 셀 5 와 같은 이유로 두 해시를 모두 찍는다 — tar 가 옛것인지 .sha256 만
    # 옛것인지 구별하려면 값이 필요하다
    print('데이터 tar 불일치 ✗')
    print('  tar 실측 ', got)
    print('  사이드카 ', want)
    print('  tar 크기 ', f'{os.path.getsize(f"{BASE}/safesound-v1.tar.gz"):,} B')
    raise AssertionError('데이터 tar 와 .sha256 이 서로 다른 버전이다')
print('sha256 OK', want[:16], '...')

os.makedirs(DATA, exist_ok=True)
with tarfile.open(f'{BASE}/safesound-v1.tar.gz') as tf:
    tf.extractall(DATA)

m = json.load(open(f'{DATA}/MANIFEST.json'))
print()
print('데이터셋', m['tag'], '/ 커밋', m['commit'][:9])
for k, v in sorted(m['counts'].items()):
    print(f'  {k:<22}{v:>7}창')
print(f'  {"합계":<22}{sum(m["counts"].values()):>7}창')

In [ ]:
# 7. 학습 전 필수 검증 — KAT(축·스케일)과 클래스 가중치. 여기서 실패하면 학습하지 말 것
import subprocess

CHECK = '''
import sys
sys.path.insert(0, "/content/ai8x-training")
sys.path.insert(0, "/content/max78000-proj/datasets")
import safesound as S
print("클래스 가중치(상수):", dict(zip(S.CLASSES, S.datasets[0]["weight"])))
print("실측 재계산       :",
      dict(zip(S.CLASSES, S.class_weights("/content/ai8x-training/data/SafeSound"))))
'''

# ⚠️ --data 를 반드시 넘긴다. 기본값은 WSL 경로라 Colab 에서는 샤드를 못 찾는데,
#    예전에는 그럴 때 "생략" 하고 통과로 보였다 (로더 검사가 한 번도 안 돌았다).
#    지금은 샤드를 못 찾으면 **실패**한다.
SHARDS = '/content/ai8x-training/data/SafeSound'
for kat in ('tools/kat_safesound.py', 'tools/kat_safesound_mel.py'):
    r = subprocess.run(f'cd /content/max78000-proj && conda run -n ai8x '
                       f'--no-capture-output python {kat} --data {SHARDS}',
                       shell=True)
    assert r.returncode == 0, f'{kat} 실패 — 전처리·로더가 어긋났다. 학습하지 말 것'

# 모델 forward 회귀 검사 — 마지막 배치(59샘플) + channels_last.
# 구성 ① 학습이 에폭 0 을 다 돌고 마지막 배치에서 view() 로 죽은 적이 있다.
# 몇 초 만에 잡히는 것을 몇 시간 뒤에 알게 되는 일을 막는다.
r = subprocess.run('cd /content/max78000-proj && conda run -n ai8x '
                   '--no-capture-output python tools/kat_models.py '
                   '--ai8x /content/ai8x-training', shell=True)
assert r.returncode == 0, 'kat_models 실패 — 모델 forward 가 깨진다. 학습하지 말 것'

open('/content/_check.py', 'w').write(CHECK)
subprocess.run('conda run -n ai8x --no-capture-output python /content/_check.py', shell=True)

In [ ]:
# 8. 기준선 학습 — 재실행하면 마지막 체크포인트에서 자동으로 이어서 간다 (진행 로그가 실시간으로 흐른다)
import glob
import os
import subprocess

os.chdir('/content/ai8x-training')
os.environ['MPLBACKEND'] = 'Agg'
PROJ = '/content/max78000-proj'


def run_live(cmd):
    '''출력을 줄 단위로 흘려보낸다.

    subprocess.run(...) 은 Colab 에서 자식 출력이 버퍼링돼 학습이 끝날 때까지
    아무것도 안 보인다. 몇 시간짜리 작업에서 진행이 안 보이는 것은 곤란하다.
    '''
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    buf = []
    for line in p.stdout:
        print(line, end='', flush=True)
        buf.append(line)
    p.wait()
    return p.returncode, ''.join(buf)

# distiller 는 logs/<name>___<타임스탬프>/ 를 만든다 → 가장 최근 것을 찾는다
ckpts = sorted(glob.glob('logs/safesound-v1___*/*checkpoint.pth.tar'),
               key=os.path.getmtime)
resume = f'--resume-from {ckpts[-1]}' if ckpts else ''
print('재개:', ckpts[-1] if ckpts else '(없음 — 처음부터)')

cmd = (
    'conda run -n ai8x --no-capture-output python train.py'
    ' --epochs 150 --optimizer Adam --lr 0.001 --wd 0'
    ' --deterministic --seed 1'
    f' --compress {PROJ}/colab/schedule_safesound.yaml'
    f' --qat-policy {PROJ}/colab/qat_policy_safesound.yaml'
    ' --model ai85safesoundnet --dataset SafeSound --device MAX78000'
    ' --confusion --batch-size 128 --workers 2'
    f' --name safesound-v1 --out-dir logs {resume}'
)
print(cmd)
print()
rc, _ = run_live(cmd)
print()
print('종료 코드', rc, '— 0 이 아니면 위 로그의 마지막 오류를 볼 것')

In [ ]:
# 9. 평가 — 창/원본 혼동행렬, 원본 부트스트랩 CI, 시간당 오경보. 출력이 흐르면서 Drive 에도 저장된다
import glob
import os
import subprocess
import time

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'


def run_live(cmd):
    '''셀 8과 같은 이유로 줄 단위로 흘려보낸다 (부트스트랩 2,000회가 시간을 쓴다).'''
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    buf = []
    for line in p.stdout:
        print(line, end='', flush=True)
        buf.append(line)
    p.wait()
    return p.returncode, ''.join(buf)


best = sorted(glob.glob('logs/safesound-v1___*/*best*.pth.tar'), key=os.path.getmtime)
assert best, 'best 체크포인트가 없다 — 셀 8을 먼저 끝낼 것'
print('평가 대상:', best[-1])
print()

# `-n ai8x` 가 빠지면 base 환경에서 돌아 ModuleNotFoundError 가 난다
cmd = (
    'conda run -n ai8x --no-capture-output python '
    '/content/max78000-proj/tools/eval_confusion.py '
    f'--checkpoint {best[-1]} --data /content/ai8x-training/data '
    '--ai8x /content/ai8x-training --hop-ms 250'
)
rc, text = run_live(cmd)

out = f'{BASE}/eval-{time.strftime("%Y%m%d-%H%M%S")}.txt'
open(out, 'w', encoding='utf-8').write(text)
print()
print('종료 코드', rc, '| 저장:', out)

## 10. 기준선 결과 분석 (셀 12~14)

첫 기준선에서 배경음 오탐 34%(시간당 4,896회)가 나왔다. **재학습 전에** 이미 있는
체크포인트로 세 가지를 확인한다.

1. 배경음 로짓 오프셋을 쓸어 오경보-재현율 곡선을 본다 → 운용 임계값 문제인지
   모델 문제인지 갈린다. **1차 결과: 곡선이 매우 완만했다** (b=−2~+8 전 구간에서
   7,279→2,623회/h, 100/10/1 전부 도달 불가). 그래서 셀 12 는 등간격 b 스윕 대신
   **마진 분위수**에서 문턱값을 집고, 남은 오경보를 **배경음 출처별로 분해**한다.
   마지막 표는 **조용한 창 쿼터 쏠림**이다 — "조용함 → siren" 을 배웠는지 본다
   (가설 근거는 `docs/results/input-stats-v1.md`, 대응은 `TASKS.md` v2 항목)
2. float `best` 와 `qat_best` 를 비교한다 → 손실이 QAT 때문인지 본다
3. 학습 곡선에서 QAT 전환 전후를 본다 → 전환이 회복됐는지 본다

In [ ]:
# 12. 오경보 문턱값 — 마진 분위수 스윕 + 배경음 출처별 분해 (+ 등간격 b 스윕)
import glob
import os
import subprocess
import time

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'


def run_live(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    buf = []
    for line in p.stdout:
        print(line, end='', flush=True)
        buf.append(line)
    p.wait()
    return p.returncode, ''.join(buf)


stamp = time.strftime('%Y%m%d-%H%M%S')

# ④와 ① **둘 다** 돌린다. 청취(2026-09-27)에서 나온 예상을 확인하려면 두 구성의
# 출처별 자체 오탐률을 나란히 봐야 한다 — 알람→siren 이 ①에서 줄어드는가.
# (사이렌은 음높이가 휘고 알람은 고정 음높이 반복 → 스펙트로그램이 유리할 것)
RUNS = [('④ wave', 'wave', 'logs/safesound-v1*___*/*qat_best*.pth.tar'),
        ('① mel', 'mel', 'logs/safesound-mel-v1*___*/*qat_best*.pth.tar')]

for label, cfg, pat in RUNS:
    ck = sorted(glob.glob(pat), key=os.path.getmtime)
    if not ck:
        print(f'{label}: 체크포인트 없음 — 건너뜀')
        continue
    print()
    print('=' * 72)
    print(f'{label}   {ck[-1]}')
    print('=' * 72)
    rc, text = run_live(
        'conda run -n ai8x --no-capture-output python '
        '/content/max78000-proj/tools/eval_threshold.py '
        f'--checkpoint {ck[-1]} --data /content/ai8x-training/data '
        f'--ai8x /content/ai8x-training --hop-ms 250 --config {cfg} '
        # b 는 테스트셋을 **원본 단위**로 반분해 한쪽에서 고르고 다른 쪽에 적용한다.
        # ai8x 검증셋(--pick-on val)은 창 단위 분리라 같은 녹음이 학습에 들어가 있어
        # 낙관적으로 나온다.
        '--pick-on test-half '
        # 마진 모드는 기본이다. 출처 태그는 샤드의 index.csv 에서 읽으므로 따로 올릴
        # 파일이 없다 (데이터셋별 분해만 매니페스트가 필요한데, 배경음이 전량
        # FSD50K 라 어차피 한 줄이다).
        f'--png {BASE}/threshold-{cfg}-{stamp}.png '
        f'--json {BASE}/threshold-{cfg}-{stamp}.json')
    open(f'{BASE}/threshold-{cfg}-{stamp}.txt', 'w',
         encoding='utf-8').write(text)
    print()
    print('종료 코드', rc)

print()
print('두 출력의 **출처별 분해** 표를 나란히 볼 것. 핵심은 하드네거티브:alarm 행이다.')
print('셀 24 가 같은 비교를 자체 오탐률 표로 자동 정리한다.')

In [ ]:
# 13. float best vs qat_best — 양자화로 얼마나 잃었는지 같은 지표로 비교한다
import glob
import json
import os
import subprocess
import time

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'
stamp = time.strftime('%Y%m%d-%H%M%S')


def newest(pat):
    f = sorted(glob.glob(pat), key=os.path.getmtime)
    return f[-1] if f else None


# qat_best 도 *best* 에 걸리므로 이름으로 갈라낸다
qat = newest('logs/safesound-v1___*/*qat_best*.pth.tar')
allbest = sorted(glob.glob('logs/safesound-v1___*/*best*.pth.tar'), key=os.path.getmtime)
flt = next((p for p in reversed(allbest) if 'qat' not in os.path.basename(p)), None)
print('float best:', flt)
print('qat   best:', qat)
assert flt and qat, 'float best 또는 qat_best 가 없다'

out = {}
for tag, ck in [('float', flt), ('qat', qat)]:
    j = f'{BASE}/eval-{tag}-{stamp}.json'
    subprocess.run(
        'conda run -n ai8x --no-capture-output python '
        '/content/max78000-proj/tools/eval_confusion.py '
        f'--checkpoint {ck} --data /content/ai8x-training/data '
        f'--ai8x /content/ai8x-training --hop-ms 250 --n-boot 500 --json {j}',
        shell=True)
    out[tag] = json.load(open(j))

names = list(out['float']['window']['recall'])
print()
print(f"{'클래스':<12}{'float 창':>10}{'qat 창':>10}{'차이':>9}   "
      f"{'float 원본':>11}{'qat 원본':>10}")
print('-' * 64)
for n in names:
    f_w = 100 * out['float']['window']['recall'][n]
    q_w = 100 * out['qat']['window']['recall'][n]
    f_o = 100 * out['float']['origin']['recall'][n]
    q_o = 100 * out['qat']['origin']['recall'][n]
    print(f'{n:<12}{f_w:>9.1f}%{q_w:>9.1f}%{q_w-f_w:>+8.1f}p   {f_o:>10.1f}%{q_o:>9.1f}%')
print()
for tag in ('float', 'qat'):
    s = out[tag]
    print(f"  {tag:<6} macro-F1 창 {s['window']['macro_f1']:.4f} / "
          f"원본 {s['origin']['macro_f1']:.4f} | "
          f"오경보 {s['false_alarm']['per_hour']:,.0f}회/h")

In [ ]:
# 14. 학습 곡선 — 에폭별 train/val Top1·loss 와 QAT 전환 전후 요약
import glob
import os
import subprocess
import time

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'
stamp = time.strftime('%Y%m%d-%H%M%S')

logs = sorted(glob.glob('logs/safesound-v1___*/*.log'), key=os.path.getmtime)
assert logs, '학습 로그가 없다'
print('로그:', logs[-1])
print()

p = subprocess.Popen(
    'conda run -n ai8x --no-capture-output python '
    '/content/max78000-proj/tools/parse_trainlog.py '
    f'--log "{logs[-1]}" --qat-epoch 60 --every 10 '
    f'--png {BASE}/curves-{stamp}.png',
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
buf = []
for line in p.stdout:
    print(line, end='', flush=True)
    buf.append(line)
p.wait()
open(f'{BASE}/curves-{stamp}.txt', 'w', encoding='utf-8').write(''.join(buf))

In [ ]:
# 15. 오분류 창을 wav 로 내보낸다 — 라벨 문제인지 모델 문제인지 귀로 가른다
#     (1) scream → siren/dog_bark 무작위 15개씩
#     (2) background → siren/dog_bark **확률 상위** 15개씩 (파일명에 출처 태그)
import glob
import os
import subprocess

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'

qat = sorted(glob.glob('logs/safesound-v1___*/*qat_best*.pth.tar'), key=os.path.getmtime)
assert qat, 'qat_best 체크포인트가 없다'

JOBS = [
    # (정답, 예측, 표본 추출, 출처 태그, 저장 위치)
    ('scream', 'random', False, f'{BASE}/listen_scream_errors'),
    # 배경음은 **확률 상위**를 뽑는다. 모델이 확신한 오경보일수록 배경음 라벨에
    # 진짜 사이렌·개 짖음이 섞여 있을 확률이 높다 — 그게 이 청취의 쟁점이다.
    ('background', 'top', True, f'{BASE}/listen_bg_errors'),
]

for true_cls, pick, tag, out in JOBS:
    print(f'--- {true_cls} → siren / dog_bark  ({pick}) ---')
    cmd = ('conda run -n ai8x --no-capture-output python '
           '/content/max78000-proj/tools/export_errors.py '
           f'--checkpoint {qat[-1]} --data /content/ai8x-training/data '
           '--ai8x /content/ai8x-training '
           f'--true {true_cls} --pred siren dog_bark --n 15 '
           f'--pick {pick} --out {out}')
    if tag:
        cmd += ' --tag-source'
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    print()
    print('Drive 에서 듣기:', out)
    print()

In [ ]:
# 16. 과소적합 판정 — train(무증강) vs test 클래스별 재현율을 나란히 본다
#
#     학습 로그의 train Top1 으로는 판정할 수 없다. 손실이 역빈도 가중(bg 0.27)
#     이라 모델이 배경음을 의도적으로 희생하는데 Top1 은 비가중이고, train Top1
#     은 증강이 걸린 데이터 기준이라 평가 조건이 아니다.
#     여기서는 **같은 체크포인트·같은 무증강 경로**로 두 split 을 돌려 비교한다.
#       · train 재현율도 낮다  → 학습 데이터조차 못 맞힌다 (용량·최적화·라벨)
#       · train 만 높다        → 일반화 문제 (데이터 양·다양성·증강)
import glob
import json
import os
import subprocess
import time

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'
stamp = time.strftime('%Y%m%d-%H%M%S')

qat = sorted(glob.glob('logs/safesound-v1___*/*qat_best*.pth.tar'), key=os.path.getmtime)
assert qat, 'qat_best 체크포인트가 없다'

out = {}
for split in ('train', 'test'):
    j = f'{BASE}/eval-{split}-{stamp}.json'
    subprocess.run(
        'conda run -n ai8x --no-capture-output python '
        '/content/max78000-proj/tools/eval_confusion.py '
        f'--checkpoint {qat[-1]} --data /content/ai8x-training/data '
        f'--ai8x /content/ai8x-training --split {split} '
        f'--hop-ms 250 --n-boot 500 --json {j}',
        shell=True)
    out[split] = json.load(open(j))

names = list(out['test']['window']['recall'])
print()
print(f"{'클래스':<12}{'train 창':>10}{'test 창':>10}{'차이':>9}   "
      f"{'train 원본':>11}{'test 원본':>11}")
print('-' * 65)
for n in names:
    tr = 100 * out['train']['window']['recall'][n]
    te = 100 * out['test']['window']['recall'][n]
    tro = 100 * out['train']['origin']['recall'][n]
    teo = 100 * out['test']['origin']['recall'][n]
    print(f'{n:<12}{tr:>9.1f}%{te:>9.1f}%{te-tr:>+8.1f}p   {tro:>10.1f}%{teo:>10.1f}%')
print()
for s in ('train', 'test'):
    d = out[s]
    print(f"  {s:<6} macro-F1 창 {d['window']['macro_f1']:.4f} / "
          f"원본 {d['origin']['macro_f1']:.4f} | "
          f"오경보 {d['false_alarm']['per_hour']:,.0f}회/h")
print()
print('  ⚠️ train split 에는 ai8x 가 떼어 간 검증 창(10%)도 섞여 있다.')
print('     과소적합 판정용 수치이지 일반화 성능이 아니다.')

## 17~20. G8 구성 ① — 로그 멜 + 2D CNN

구성 ④(raw 파형 1D CNN)의 대조군이다. **입력 표현만 다르고 나머지는 전부 같다** —
같은 샤드·같은 원본 단위 분할·같은 클래스 가중치·같은 증강·같은 스케줄·같은 QAT·
같은 평가 스크립트.

| | 구성 ④ (wave) | 구성 ① (mel) |
|---|---|---|
| 입력 | raw 파형 (128, 128) int8 | 로그 멜 (1, 64, 64) int8 |
| CPU 가 하는 일 | 없음 | STFT + 멜 + 로그 |
| params | 165,457 (36.6%) | 157,535 (34.8%) |

**①은 ④의 병목이 입력 표현인지 데이터인지 가리는 실험이다.** 기준선이 과소적합
쪽인지 아닌지와 무관하게 지금 돌린다 — ①도 같이 낮으면 데이터·라벨 문제이고,
①만 잘 나오면 raw 파형 표현이 병목이라는 뜻이다.

셀 구성: **17** 특징 캐시 / **18** ① 학습(시드 1) / **19** 반복 ④ wave / **20** 반복 ① mel /
**21** B KWS20 초기화 / **22** D-1 필터뱅크 초기화 /
**23** wave2D 대조(fold) / **24** 비교 (평균±편차).
실행 순서는 목표에 직접 닿는 것부터다 — B → D-1 → wave2D.

**첫 결과 (시드 1, 2026-09-27)**: 고정 오경보 300회/h 에서 macro-F1
④ 0.387 vs ① 0.673. `glass` +42.6 / `scream` +35.6 / `dog_bark` +39.4 %p,
`siren` +2.1%p(잡음 안).
⚠️ 단일 시드이고, ①과 ④는 표현뿐 아니라 **구조도** 다르다 — 결과와 한계는
`docs/results/g8-first-comparison.md` 를 먼저 읽을 것.

설계 근거: `docs/results/config1-mel2d-design.md`,
양자화 구간 근거: `docs/results/mel-range-sweep.md`.

In [ ]:
# 17. 구성 ① 특징 캐시 — 테스트셋은 무증강이라 한 번만 계산해 두면 평가가 빨라진다
#     (학습셋은 매 에폭 증강이 달라야 하므로 캐시하지 않는다 — 코드가 막아 둔다)
import os
import subprocess

BUILD = '''
import os, sys, time
sys.path.insert(0, "/content/ai8x-training")
sys.path.insert(0, "/content/max78000-proj/datasets")
import melfeat as MF, safesound_mel as SM
root = "/content/ai8x-training/data/SafeSound"
path = os.path.join(root, "test", f"melcache_{MF.N_MELS}x{MF.N_FRAMES}.npy")
if os.path.isfile(path):
    print("캐시 이미 있음:", path)
else:
    ds = SM.SafeSoundMel(root, "test", transform=None, augment=False)
    t0 = time.time()
    ds.build_cache(path)
    print(f"캐시 생성 {len(ds):,}창  {time.time()-t0:.0f}s  ->", path)
'''
open('/content/_melcache.py', 'w').write(BUILD)
subprocess.run('conda run -n ai8x --no-capture-output python /content/_melcache.py',
               shell=True)

In [ ]:
# 18. 구성 ① 학습 — ④와 **같은 스케줄·같은 QAT·같은 에폭**. 재실행하면 이어서 간다
import glob
import os
import subprocess

os.chdir('/content/ai8x-training')
os.environ['MPLBACKEND'] = 'Agg'
PROJ = '/content/max78000-proj'


def run_live(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    return p.returncode


ckpts = sorted(glob.glob('logs/safesound-mel-v1___*/*checkpoint.pth.tar'),
               key=os.path.getmtime)
resume = f'--resume-from {ckpts[-1]}' if ckpts else ''
print('재개:', ckpts[-1] if ckpts else '(없음 — 처음부터)')

# ⚠️ 아래 인자는 셀 8(구성 ④)과 **한 글자도 다르면 안 된다** — 모델·데이터셋·
#    이름만 빼고. 다르면 비교가 오염된다.
cmd = (
    'conda run -n ai8x --no-capture-output python train.py'
    ' --epochs 150 --optimizer Adam --lr 0.001 --wd 0'
    ' --deterministic --seed 1'
    f' --compress {PROJ}/colab/schedule_safesound.yaml'
    f' --qat-policy {PROJ}/colab/qat_policy_safesound.yaml'
    ' --model ai85safesoundmelnet --dataset SafeSoundMel --device MAX78000'
    ' --confusion --batch-size 128 --workers 2'
    f' --name safesound-mel-v1 --out-dir logs {resume}'
)
print(cmd)
print()
print('종료 코드', run_live(cmd))

In [ ]:
# 19. 반복 실험 ④ wave — **맨 위 SEED 한 줄만 바꿔 다시 누른다**
SEED = 2

# ⚠️ 시드 1 은 셀 8 이 이름 없이 돌렸다(`safesound-v1`). 여기서는 `-s<시드>` 를
#    붙이므로 **기존 시드 1 결과를 건드리지 않는다.** 셀 21 의 글롭이 둘 다 잡고,
#    접미사 없는 이름은 시드 1 로 읽는다.
# ⚠️ --seed / --name 을 뺀 나머지 인자는 셀 8 과 **한 글자도 다르지 않다.**
#    비교의 전제가 "설정이 같다" 이기 때문이다.
import glob
import os
import subprocess

os.chdir('/content/ai8x-training')
os.environ['MPLBACKEND'] = 'Agg'
PROJ = '/content/max78000-proj'
NAME = 'safesound-v1' if SEED == 1 else f'safesound-v1-s{SEED}'


def run_live(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    return p.returncode


ck = sorted(glob.glob(f'logs/{NAME}___*/*checkpoint.pth.tar'), key=os.path.getmtime)
resume = f'--resume-from {ck[-1]}' if ck else ''
print(f'④ wave  시드 {SEED}  이름 {NAME}')
print('재개:', ck[-1] if ck else '(없음 — 처음부터)')
print()

rc = run_live(
    'conda run -n ai8x --no-capture-output python train.py'
    ' --epochs 150 --optimizer Adam --lr 0.001 --wd 0'
    f' --deterministic --seed {SEED}'
    f' --compress {PROJ}/colab/schedule_safesound.yaml'
    f' --qat-policy {PROJ}/colab/qat_policy_safesound.yaml'
    ' --model ai85safesoundnet --dataset SafeSound --device MAX78000'
    ' --confusion --batch-size 128 --workers 2'
    f' --name {NAME} --out-dir logs {resume}')
print()
print('종료 코드', rc, '— 0 이 아니면 위 로그의 마지막 오류를 볼 것')

In [ ]:
# 20. 반복 실험 ① mel — **맨 위 SEED 한 줄만 바꿔 다시 누른다**
SEED = 2

# 셀 19 와 같은 규약이다. 시드 1 은 셀 18 이 `safesound-mel-v1` 로 돌렸고,
# 여기서는 `-s<시드>` 를 붙인다. 나머지 인자는 셀 18 과 한 글자도 다르지 않다.
import glob
import os
import subprocess

os.chdir('/content/ai8x-training')
os.environ['MPLBACKEND'] = 'Agg'
PROJ = '/content/max78000-proj'
NAME = 'safesound-mel-v1' if SEED == 1 else f'safesound-mel-v1-s{SEED}'


def run_live(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    return p.returncode


ck = sorted(glob.glob(f'logs/{NAME}___*/*checkpoint.pth.tar'), key=os.path.getmtime)
resume = f'--resume-from {ck[-1]}' if ck else ''
print(f'① mel  시드 {SEED}  이름 {NAME}')
print('재개:', ck[-1] if ck else '(없음 — 처음부터)')
print()

rc = run_live(
    'conda run -n ai8x --no-capture-output python train.py'
    ' --epochs 150 --optimizer Adam --lr 0.001 --wd 0'
    f' --deterministic --seed {SEED}'
    f' --compress {PROJ}/colab/schedule_safesound.yaml'
    f' --qat-policy {PROJ}/colab/qat_policy_safesound.yaml'
    ' --model ai85safesoundmelnet --dataset SafeSoundMel --device MAX78000'
    ' --confusion --batch-size 128 --workers 2'
    f' --name {NAME} --out-dir logs {resume}')
print()
print('종료 코드', rc, '— 0 이 아니면 위 로그의 마지막 오류를 볼 것')

In [ ]:
# 21. B. KWS20 v3 사전학습 초기화 + 파인튜닝 — **맨 위 상수만 바꾼다**
#
#     ④는 KWS20 v3 파생이라 conv 8층 가중치 모양이 전부 같다. FC 만 21→5 로
#     새로 초기화하고 나머지를 물려받는다. 체크포인트는 ai8x-synthesis 레포의
#     trained/ 에 들어 있다 (따로 받을 것 없음).
#
#     ⚠️ 가장 큰 위험은 **입력 스케일**이다. kws20.py:392 가 파일 단위 피크
#        정규화를 하고 우리는 하지 않는다 (CLAUDE.md 7장). 사전학습 1층 필터는
#        피크가 일정한 입력에 맞춰져 있다. 랜덤 게인 ±12dB 가 흡수해 주길
#        기대하지만 안 될 수도 있다. **정규화 도입은 금지**다.
#     ⚠️ 중단 기준: **30에폭 안에 val Top1 이 48.23% 를 못 넘으면** 스케일
#        불일치로 보고 중단한다. 48.23% 는 ④ 기준선(시드 1) 로그의 epoch 30
#        val Top1 이다 — 같은 에폭에서 사전학습이 이기지 못하면 출발점이
#        도움이 안 된다는 뜻이다.
#
#     ⚠️ **이 실행은 초기화 외에도 LR·스케줄이 기준선과 다르다.** 비교표에
#        "파인튜닝 설정" 으로 함께 적을 것 — 아래 셀 출력이 그 항목을 찍는다.
SEED = 1
LR = 0.0003          # 파인튜닝이라 기준선(0.001)보다 낮춘다
BASELINE_E30 = 48.23  # ④ 기준선 시드 1 의 epoch 30 val Top1 (%) — 중단 기준 비교값
PRETRAINED = '/content/ai8x-synthesis/trained/ai85-kws20_v3-qat8.pth.tar'

import glob
import os
import subprocess

os.chdir('/content/ai8x-training')
os.environ['MPLBACKEND'] = 'Agg'
PROJ = '/content/max78000-proj'
INIT = '/content/kws20v3-init-5cls.pth.tar'
NAME = 'safesound-kws20init-v1' if SEED == 1 else f'safesound-kws20init-v1-s{SEED}'


def run_live(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    return p.returncode


# ai8x-synthesis 를 받지 않았으면 체크포인트만 있으면 된다. 없으면 클론한다.
if not os.path.isfile(PRETRAINED):
    print('사전학습 체크포인트가 없다 — ai8x-synthesis 를 클론한다 (얕게)')
    subprocess.run('git clone --depth 1 '
                   'https://github.com/analogdevicesinc/ai8x-synthesis.git '
                   '/content/ai8x-synthesis', shell=True)
assert os.path.isfile(PRETRAINED), f'없다: {PRETRAINED}'

# 1) 가중치를 우리 구조에 얹는다. **로드된 층 / 새로 초기화된 층이 표로 찍힌다.**
rc = run_live('conda run -n ai8x --no-capture-output python '
              f'{PROJ}/tools/init_from_kws20.py --src {PRETRAINED} '
              f'--out {INIT} --ai8x /content/ai8x-training')
assert rc == 0, 'init_from_kws20 실패'

# 2) 파인튜닝. --resume-from 이 아니라 --exp-load-weights-from 이다
#    (에폭·옵티마이저 상태를 물려받으면 파인튜닝이 아니게 된다).
ck = sorted(glob.glob(f'logs/{NAME}___*/*checkpoint.pth.tar'), key=os.path.getmtime)
resume = f'--resume-from {ck[-1]}' if ck else f'--exp-load-weights-from {INIT}'
print()
print(f'B  시드 {SEED}  LR {LR}  이름 {NAME}')
print('시작점:', ck[-1] if ck else INIT)
print()
print('=== 기준선과 다른 점 (비교표에 "파인튜닝 설정" 으로 적을 것) ===')
print(f'  1) 초기화   KWS20 v3 사전학습 (conv 8층 + bias)  ← 이 실험의 목적')
print(f'  2) LR       {LR} (기준선 0.001)')
print(f'  3) 스케줄   schedule_safesound_ft.yaml — 첫 감쇠 40 (기준선 75)')
print(f'  4) bias     True (기준선 False) — 체크포인트의 conv bias 를 받기 위해')
print(f'  총 에폭 150 / QAT 시작 60 은 기준선과 같다 (비교 가능성 유지)')
print()
print('=== 중단 기준 ===')
print(f'  epoch 30 의 val Top1 이 **{BASELINE_E30:.2f}%** 를 넘어야 한다')
print(f'  (④ 기준선 시드 1 의 epoch 30 val Top1). 못 넘으면 입력 스케일')
print(f'  불일치로 보고 중단 — kws20 은 파일 단위 피크 정규화를 하고 우리는')
print(f'  하지 않는다 (CLAUDE.md 7장). 정규화 도입으로 맞추는 것은 금지다.')
print()

rc = run_live(
    'conda run -n ai8x --no-capture-output python train.py'
    f' --epochs 150 --optimizer Adam --lr {LR} --wd 0'
    f' --deterministic --seed {SEED}'
    f' --compress {PROJ}/colab/schedule_safesound_ft.yaml'
    f' --qat-policy {PROJ}/colab/qat_policy_safesound.yaml'
    ' --model ai85safesoundnet_bias --dataset SafeSound --device MAX78000'
    ' --confusion --batch-size 128 --workers 2'
    f' --name {NAME} --out-dir logs {resume}')
print()
print('종료 코드', rc)
print()
print('⚠️ 비교표에 반드시 적을 것 — 이 실행은 **초기화 + LR + 스케줄 + bias** 가')
print('   기준선과 다르다. "사전학습 초기화의 효과" 라고만 적으면 틀린 서술이다.')
print('   LR·스케줄만 바꾼 대조가 필요하면 셀 19(④ 반복)에서 LR 을 0.0003 으로,')
print('   --compress 를 schedule_safesound_ft.yaml 로 바꿔 한 번 더 돌린다.')

In [ ]:
# 22. D-1. 필터뱅크 초기화 — **G8 구성 ③ (학습 프론트엔드)**. 맨 위 상수만 바꾼다
#
#     ④의 첫 층 voice_conv1 은 Conv1d(128→100, k=1) 이고, 입력 채널 축이
#     프레임 내 잔 시간축이라 **길이 128 FIR 100개와 수학적으로 같다.**
#     여기에 멜 대역통과 필터를 넣으면 전처리를 네트워크 안에 심은 것이 된다.
#
#     cos/sin 쌍 + **Abs** 활성화로 |re|, |im| 을 얻는다 → 밴드당 2필터, 50밴드.
#     ReLU 면 부호 쌍까지 필요해 25밴드로 준다 (USE_RELU=True 가 그 판).
#     자세한 근거는 tools/init_filterbank.py 의 설명.
#
#     ⚠️ FREEZE=False(기본)를 **먼저** 돌린다. 기준선 대비 바뀌는 것이 초기값
#        뿐이라 원인이 깨끗하고, 구성 ③의 정의가 "학습" 프론트엔드다.
#        고정 판은 학습 가능 판 결과를 본 뒤 두 번째 실험으로 돌린다.
SEED = 1
LR = 0.0003          # 초기값이 의미 있는 출발점이므로 낮게 (B 와 같은 이유)
USE_RELU = False     # True 면 ReLU 유지 + 25밴드 (초기화만 바꾸는 순수 대조)
FREEZE = False       # True 면 첫 층 고정 (두 번째 실험)

import glob
import os
import subprocess

os.chdir('/content/ai8x-training')
os.environ['MPLBACKEND'] = 'Agg'
PROJ = '/content/max78000-proj'
INIT = '/content/fb-init.pth.tar'
MODEL = 'ai85safesoundnet_fb_relu' if USE_RELU else 'ai85safesoundnet_fb'
TAG = ('fb' + ('relu' if USE_RELU else 'abs') + ('frz' if FREEZE else ''))
NAME = f'safesound-{TAG}-v1' if SEED == 1 else f'safesound-{TAG}-v1-s{SEED}'


def run_live(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    return p.returncode


# 1) 필터뱅크 초기 체크포인트. 톤 검산 결과가 함께 찍힌다 — 기대 밴드가
#    켜지지 않으면 접기 방향이 틀린 것이므로 거기서 멈춰야 한다.
opt = (' --relu' if USE_RELU else '') + (' --freeze-conv1' if FREEZE else '')
rc = run_live('conda run -n ai8x --no-capture-output python '
              f'{PROJ}/tools/init_filterbank.py --out {INIT} '
              f'--ai8x /content/ai8x-training{opt}')
assert rc == 0, 'init_filterbank 실패'

# 2) 학습
ck = sorted(glob.glob(f'logs/{NAME}___*/*checkpoint.pth.tar'), key=os.path.getmtime)
resume = f'--resume-from {ck[-1]}' if ck else f'--exp-load-weights-from {INIT}'
print()
print(f'D-1  시드 {SEED}  LR {LR}  활성화 {"ReLU" if USE_RELU else "Abs"}  '
      f'고정 {FREEZE}  이름 {NAME}')
print('시작점:', ck[-1] if ck else INIT)
print()

rc = run_live(
    'conda run -n ai8x --no-capture-output python train.py'
    f' --epochs 150 --optimizer Adam --lr {LR} --wd 0'
    f' --deterministic --seed {SEED}'
    f' --compress {PROJ}/colab/schedule_safesound_ft.yaml'
    f' --qat-policy {PROJ}/colab/qat_policy_safesound.yaml'
    f' --model {MODEL} --dataset SafeSound --device MAX78000'
    ' --confusion --batch-size 128 --workers 2'
    f' --name {NAME} --out-dir logs {resume}')
print()
print('종료 코드', rc)

In [ ]:
# 23. wave2D 대조 — **fold 판이 기본값**. 맨 위 상수만 바꾼다
#
#     ①과 같은 2D 구조에 ④와 같은 raw 파형을 넣어 격차의 원인을 가른다.
#     **`ai8x.fold(2)` 로 (1,128,128) → (4,64,64)** 로 접는다 — 인터레이스
#     부분표본이라 **버리는 값이 하나도 없고**(왕복 검증 완료), 2×2 블록의 4개
#     위상이 서로 다른 채널로 분리되어 3×3 conv 가 함께 본다.
#     파라미터 158,399 (①의 +0.5%), 채널당 4,096픽셀로 HWC 한계 8,192 안.
#
#     ⚠️ `USE_POOL=True` 는 선행 2×2 AvgPool 판이다. 그 평균이 접기의 행(연속
#        샘플) 축에서 2탭 저역통과 + 2배 데시메이션이라 **4kHz 이상을 버린다**
#        (유리 파손에 직접 불리). 비교가 필요할 때만 쓴다.
#
#     ⚠️ 이름표는 **"구조 변경, 입력 정보 보존"** 이다. "표현만" 이 아니다 —
#        (128,128) 접기 자체가 스펙트로그램이 아니고(행은 연속 샘플, 열은 8ms),
#        1D conv 대 2D conv 라는 연산 차이도 남는다.
#        docs/results/g8-first-comparison.md 4.1
SEED = 1
USE_POOL = False     # 기본 False = fold 판 (정보 손실 없음)

import glob
import os
import subprocess

os.chdir('/content/ai8x-training')
os.environ['MPLBACKEND'] = 'Agg'
PROJ = '/content/max78000-proj'
if USE_POOL:
    MODEL, DATASET, TAG = 'ai85safesoundwave2dnet', 'SafeSoundWave2D', 'w2dpool'
else:
    MODEL, DATASET, TAG = ('ai85safesoundwave2dfoldnet', 'SafeSoundWave2DFold',
                           'w2d')
NAME = f'safesound-{TAG}-v1' if SEED == 1 else f'safesound-{TAG}-v1-s{SEED}'


def run_live(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    return p.returncode


ck = sorted(glob.glob(f'logs/{NAME}___*/*checkpoint.pth.tar'), key=os.path.getmtime)
resume = f'--resume-from {ck[-1]}' if ck else ''
print(f'wave2D ({"AvgPool" if USE_POOL else "fold"})  시드 {SEED}  이름 {NAME}')
print('재개:', ck[-1] if ck else '(없음 — 처음부터)')
print()

# --seed / --model / --dataset / --name 외 인자는 셀 8·18 과 한 글자도 다르지 않다
rc = run_live(
    'conda run -n ai8x --no-capture-output python train.py'
    ' --epochs 150 --optimizer Adam --lr 0.001 --wd 0'
    f' --deterministic --seed {SEED}'
    f' --compress {PROJ}/colab/schedule_safesound.yaml'
    f' --qat-policy {PROJ}/colab/qat_policy_safesound.yaml'
    f' --model {MODEL} --dataset {DATASET} --device MAX78000'
    ' --confusion --batch-size 128 --workers 2'
    f' --name {NAME} --out-dir logs {resume}')
print()
print('종료 코드', rc, '— 0 이 아니면 위 로그의 마지막 오류를 볼 것')

In [ ]:
# 24. 구성 비교 — 시드별 결과를 모아 **평균±편차**로 낸다. G8 의 최소 버전이다
#
#     ④ wave (파형 1D) / ① mel (멜 2D) / wave2D (파형 2D) 중 체크포인트가 있는
#     것을 전부 찾아 평가하고 구성별로 평균±표준편차를 낸다. 시드 1 은 접미사
#     없는 이름이고 시드 2 이상은 `-s<시드>` 다.
#
#     **비교 기준은 고정 오경보 지점이다.** argmax 는 각 모델이 알아서 정한
#     동작점이라, 배경음을 더 버린 모델이 이벤트 재현율만 높게 보인다.
#
#     목표선: ① mel 의 300/h macro-F1 = 0.673. 성공 기준은 그 90% ≈ 0.606.
import glob
import json
import os
import re
import subprocess
import time

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'
stamp = time.strftime('%Y%m%d-%H%M%S')

CFGS = [('④ wave', 'wave', 'safesound-v1'),
        ('① mel', 'mel', 'safesound-mel-v1'),
        ('wave2D', 'wave2dfold', 'safesound-w2d-v1'),
        ('w2D pool', 'wave2d', 'safesound-w2dpool-v1'),
        # ④ 살리기 실험들. 체크포인트가 있는 것만 표에 들어간다
        # ⚠️ B 는 초기화 외에 LR 0.0003 · 파인튜닝 스케줄 · bias=True 도 다르다.
        #    이름에 (ft) 를 달아 표에서 그 사실이 보이게 한다
        ('B kws20(ft)', 'wave_bias', 'safesound-kws20init-v1'),
        ('D1 abs', 'wave_fb', 'safesound-fbabs-v1'),
        ('D1 relu', 'wave_fb_relu', 'safesound-fbrelu-v1'),
        ('D1 frz', 'wave_fb', 'safesound-fbabsfrz-v1')]
TARGET = 0.673          # ① mel 300/h macro-F1 (시드 1). 목표선
BAR = 0.90 * TARGET     # 성공 기준


def mean_sd(vals):
    '''(평균, 표준편차, n). n=1 이면 편차는 None 이다 — 0 이 아니라 모르는 것이다.'''
    n = len(vals)
    if n == 0:
        return None, None, 0
    m = sum(vals) / n
    if n == 1:
        return m, None, 1
    var = sum((v - m) ** 2 for v in vals) / (n - 1)      # 표본 표준편차
    return m, var ** 0.5, n


def pct(m, sd, w=14):
    if m is None:
        return f"{'—':>{w}}"
    s = f'{100*m:.1f}' + (f'±{100*sd:.1f}' if sd is not None else '') + '%'
    return f'{s:>{w}}'


def f4(m, sd):
    if m is None:
        return '—'
    return f'{m:.4f}' + (f'±{sd:.4f}' if sd is not None else '')


# ── 체크포인트를 구성×시드로 모아 평가한다
runs = {}                          # label -> {seed: json}
for label, cfg, base in CFGS:
    for d in sorted(glob.glob(f'logs/{base}___*') + glob.glob(f'logs/{base}-s*___*')):
        m = re.search(r'-s(\d+)___', d)
        seed = int(m.group(1)) if m else 1
        ck = sorted(glob.glob(f'{d}/*qat_best*.pth.tar'), key=os.path.getmtime)
        if not ck:
            continue
        j = f'{BASE}/eval-g8-{cfg}-s{seed}-{stamp}.json'
        print(f'--- {label} 시드 {seed}: {os.path.basename(ck[-1])}')
        subprocess.run(
            'conda run -n ai8x --no-capture-output python '
            '/content/max78000-proj/tools/eval_confusion.py '
            f'--checkpoint {ck[-1]} --data /content/ai8x-training/data '
            f'--ai8x /content/ai8x-training --config {cfg} '
            f'--hop-ms 250 --n-boot 2000 --json {j}', shell=True)
        runs.setdefault(label, {})[seed] = json.load(open(j))

have = [lb for lb, _c, _b in CFGS if lb in runs]
if not have:
    print()
    print('체크포인트가 없다. 셀 8 / 18 / 19 / 20 / 21 을 볼 것.')
else:
    names = list(next(iter(runs[have[0]].values()))['window']['recall'])
    print()
    for lb in have:
        print(f'  {lb}: 시드 {sorted(runs[lb])}  ({len(runs[lb])}회)')

    # ── 고정 오경보 지점 (비교의 기준)
    for pt in ('1000/h', '300/h'):
        print()
        print(f'=== 고정 오경보 {pt} — 평균±표준편차 ===')
        print(f"{'클래스':<12}" + ''.join(f'{lb:>14}' for lb in have))
        print('-' * (12 + 14 * len(have)))
        # 구성이 늘면 줄이 길어진다. 터미널 폭을 넘으면 JSON 을 직접 볼 것.
        agg = {}
        for lb in have:
            agg[lb] = {}
            for n in names:
                agg[lb][n] = mean_sd(
                    [d['fixed_fa']['targets'][pt]['recall'][n]
                     for d in runs[lb].values()
                     if (d.get('fixed_fa') or {}).get('targets', {}).get(pt)])
            agg[lb]['_f1'] = mean_sd(
                [d['fixed_fa']['targets'][pt]['macro_f1']
                 for d in runs[lb].values()
                 if (d.get('fixed_fa') or {}).get('targets', {}).get(pt)])
        if any(agg[lb]['_f1'][0] is None for lb in have):
            print(f'  ({pt} 지점 없음 — 코드 번들이 오래된 것일 수 있다)')
            continue
        for n in names:
            print(f'{n:<12}' + ''.join(pct(*agg[lb][n][:2]) for lb in have))
        print(f"{'macro-F1':<12}"
              + ''.join(f'{f4(*agg[lb]["_f1"][:2]):>14}' for lb in have))
        if pt == '300/h':
            print(f'  목표선 {TARGET:.3f} (① mel 시드 1) / 성공 기준 {BAR:.3f} (90%)')
            for lb in have:
                m = agg[lb]['_f1'][0]
                mark = '달성' if m >= BAR else f'미달 ({BAR - m:+.3f})'
                print(f'    {lb:<8}{m:.4f}  {mark}')

        # 쌍별 격차 — 무엇이 분리되는지 이름을 붙여 찍는다
        # ⚠️ 이름표를 과장하지 않는다. wave2D 는 "구조만" 이 아니다 — 128×128
        #    입력에 선행 2×2 AvgPool 이 붙고(채널당 8,192픽셀 한계로 불가피),
        #    그 풀링이 연속 샘플 축의 4kHz 이상을 버리고 8ms 떨어진 프레임을
        #    섞는다. 그래서 '구조 변경(선행 풀링 포함)' 이라고 적는다.
        PAIRS = [('① mel', '④ wave', '표현 + 구조 (지금까지 가진 것)'),
                 ('① mel', 'wave2D', '표현 (구조·풀링 고정)'),
                 ('wave2D', '④ wave', '구조 변경, 입력 정보 보존')]
        shown = [(x, y, why) for x, y, why in PAIRS if x in have and y in have]
        if shown:
            print(f'  쌍별 macro-F1 격차 ({pt})')
            for x, y, why in shown:
                print(f'    {x} − {y} = {agg[x]["_f1"][0] - agg[y]["_f1"][0]:+.4f}'
                      f'   ← {why}')
            if len(shown) == 3:
                print('    셋이 다 있으면 격차를 두 갈래로 나눌 수 있다.')
                print('    fold 판은 입력 정보를 버리지 않으므로 풀링 혼입이')
                print('    없다. 다만 (128,128) 접기가 스펙트로그램이 아니고')
                print('    1D/2D conv 차이가 남아 "표현만" 은 아니다.')
                print('    docs/results/g8-first-comparison.md 4.1')

    # ── argmax (참고용, 비교 근거로 쓰지 않는다)
    print()
    print('=== argmax — 참고용. 동작점이 모델마다 달라 비교 근거가 아니다 ===')
    for lb in have:
        w = mean_sd([d['window']['macro_f1'] for d in runs[lb].values()])
        o = mean_sd([d['origin']['macro_f1'] for d in runs[lb].values()])
        fa = mean_sd([d['false_alarm']['per_hour'] for d in runs[lb].values()])
        print(f'  {lb:<8} macro-F1 창 {f4(*w[:2])} / 원본 {f4(*o[:2])} | '
              f'오경보 {fa[0]:,.0f}'
              + (f'±{fa[1]:,.0f}' if fa[1] is not None else '') + '회/h')

    # ── 출처별 자체 오탐률.
    #    청취(2026-09-27)에서 배경음 오경보의 두 축이 갈렸다:
    #      알람 → siren  (음높이 궤적 차이 → 스펙트로그램이 유리할 것으로 예상)
    #      말소리 → dog_bark (배경음에 음성이 없어서 → 입력 표현과 무관할 것)
    tags = set()
    for lb in have:
        for d in runs[lb].values():
            tags |= set(d.get('bg_sources') or {})
    if tags:
        print()
        print('=== 배경음 출처별 **→siren** 비율 (argmax) ===')
        print(f"{'출처':<26}{'창':>7}" + ''.join(f'{lb:>14}' for lb in have))
        print('-' * (33 + 14 * len(have)))
        for tag in sorted(tags):
            cells, n_win = [], 0
            for lb in have:
                ds = [d['bg_sources'][tag] for d in runs[lb].values()
                      if tag in (d.get('bg_sources') or {})]
                if ds:
                    n_win = ds[0]['n']
                cells.append(mean_sd([x['by_class_rate']['siren'] for x in ds]))
            print(f'{tag:<26}{n_win:>7,}'
                  + ''.join(pct(*c[:2]) for c in cells))
        print()
        print('  ★ 핵심은 `하드네거티브:alarm → siren` 행이다. 사이렌은 음높이가')
        print('    휘고 알람은 고정 음높이 반복이라, 스펙트로그램에서는 기울어진')
        print('    선 대 수평선이 된다 → ①에서 줄어들 것으로 예상한다.')
        print('  ☆ 대조: 말소리→dog_bark 는 배경음에 음성이 없어서 나는 것이라')
        print('    입력 표현과 무관할 것으로 본다 (v2 (b) LibriSpeech 추가 대상).')
        print('    근거: docs/results/listening-verification.md 7.2')
    print()
    print('  ⚠️ 지연·에너지는 여기서 나오지 않는다 — 보드가 있어야 한다.')
    print('     이 표는 G8 의 **정확도 축**이고, 시간 축은 Phase 5 에서 채운다.')
    print()
    print('  === 구성별로 기준선(④)과 다른 점 — 표를 읽을 때 함께 볼 것 ===')
    print('    ① mel        입력 표현(로그 멜) + 2D 구조')
    print('    wave2D       2D 구조 (입력 정보 보존 — fold 접기)')
    print('    w2D pool     2D 구조 + 선행 AvgPool (**4kHz 이상 손실**)')
    print('    B kws20(ft)  사전학습 초기화 + **LR 0.0003 + 파인튜닝 스케줄**')
    print('                 + bias=True  ← 네 가지가 함께 다르다')
    print('    D1 abs/relu  첫 층 필터뱅크 초기화 (abs 판은 활성화도 Abs)')
    print('                 + LR 0.0003 + 파인튜닝 스케줄')
    print('    D1 frz       위에 더해 첫 층 고정')
    print('  따라서 B·D1 의 격차를 "초기화의 효과" 라고만 적으면 틀린 서술이다.')

## 25. 학습이 끝난 뒤 (WSL2 에서)

혼동행렬의 **N프레임 오경보는 하한(낙관치)** 이라 논문에 쓰지 않는다. 논문 수치는
원본 오디오를 빈틈없이 훑는 스트리밍 평가에서 나오고, 그건 원본이 있는 WSL2 에서
돌리는 편이 간단하다.

```bash
# WSL2 — best 체크포인트를 Drive 에서 내려받은 뒤
~/ai8x-training/venv/bin/python tools/stream_eval.py \
  --checkpoint <best.pth.tar> --ai8x ~/ai8x-training \
  --json docs/results/stream-eval-pc.json
```

그다음 양자화·합성으로 넘어간다 (442KB / 512KB 초과 여부 확인).

**읽는 순서**: 혼동행렬(원본 단위) → 클래스별 재현율·CI → 시간당 오경보.
전체 정확도는 보지 않는다.